## 1. Импорты

In [ ]:
import os
import json
import random
import time

import torch
from PIL import Image
from IPython.display import display
from torchvision.models import ResNet34_Weights, ResNet152_Weights, resnet34, resnet152

## 2. Инициализация

In [ ]:
device = torch.device("cuda")
folder = "data/imagenet_sample"
torch.hub.set_dir("data/torch")

weights34 = ResNet34_Weights.IMAGENET1K_V1
weights152 = ResNet152_Weights.IMAGENET1K_V2
classes = weights34.meta["categories"]

models = {
    "ResNet34": resnet34(weights=weights34).to(device).eval(),
    "ResNet152": resnet152(weights=weights152).to(device).eval(),
}
transforms = {
    "ResNet34": weights34.transforms(),
    "ResNet152": weights152.transforms(),
}

print(torch.cuda.get_device_name(0))

In [ ]:
models["ResNet34"]

## 3. Изображения и правильные классы

In [ ]:
image_paths = [
    os.path.join(folder, filename)
    for filename in sorted(os.listdir(folder))
    if filename.lower().endswith((".jpg", ".jpeg"))
]

with open(os.path.join(folder, "manifest.json"), encoding="utf-8") as file:
    samples = json.load(file)

labels = {sample["filename"]: sample["class_id"] for sample in samples}
print(f"Найдено изображений: {len(image_paths)}")

## 4. Функция оценки модели

In [ ]:
def run_model(model, image_paths, labels, transform, device):
    model.eval()
    correct = 0
    total_loss = 0
    predictions = {}
    total_inference_ms = 0
    start_event = torch.cuda.Event(enable_timing=True)
    end_event = torch.cuda.Event(enable_timing=True)

    with torch.inference_mode():
        torch.cuda.synchronize(device)
        start_time = time.perf_counter()

        for path in image_paths:
            with Image.open(path) as source:
                image = source.convert("RGB")
            tensor = transform(image).unsqueeze(0).to(device)
            target = torch.tensor([labels[os.path.basename(path)]], device=device)

            start_event.record()
            output = model(tensor)
            end_event.record()
            end_event.synchronize()
            inference_ms = start_event.elapsed_time(end_event)
            total_inference_ms += inference_ms
            loss = torch.nn.functional.cross_entropy(output, target)
            probabilities = output.softmax(dim=1)[0]
            scores, indices = probabilities.topk(5)

            correct += (output.argmax(dim=1) == target).sum().item()
            total_loss += loss.item()
            predictions[path] = {
                "indices": indices.tolist(),
                "scores": scores.tolist(),
                "cross_entropy": loss.item(),
                "inference_ms": inference_ms,
            }

    total_time = time.perf_counter() - start_time
    return {
        "accuracy": correct / len(image_paths),
        "cross_entropy": total_loss / len(image_paths),
        "predictions": predictions,
        "total_time": total_time,
        "mean_inference_ms": total_inference_ms / len(image_paths),
    }

## 5. Сравнение моделей
_____

In [ ]:
results = {}

for name, model in models.items():
    metrics = run_model(model, image_paths, labels, transforms[name], device)
    results[name] = metrics
    print(name)
    print("Accuracy:", round(metrics["accuracy"] * 100, 1), "%")
    print("Cross-entropy:", round(metrics["cross_entropy"], 4))
    print("Общее время:", round(metrics["total_time"], 2), "с")
    print("Среднее время модели на фото:", round(metrics["mean_inference_ms"], 2), "мс")
    print()

## 6. Фотографии и результаты обеих моделей
______

In [ ]:
for path in random.sample(image_paths, min(5, len(image_paths))):
    filename = os.path.basename(path)
    target = labels[filename]

    print(f"{filename} — правильный класс: {classes[target]}")
    with Image.open(path) as source:
        image = source.convert("RGB")
    image.thumbnail((320, 240))
    display(image)

    for name in models:
        prediction = results[name]["predictions"][path]
        status = "верно" if prediction["indices"][0] == target else "ошибка"
        print(f"{name}: {status}, cross-entropy: {prediction['cross_entropy']:.4f}, "
                f"время модели: {prediction['inference_ms']:.2f} мс")

        for score, index in zip(prediction["scores"], prediction["indices"]):
            print(f"  {classes[index]}: {score:.1%}")
        print()

    print("—" * 60)